# **Installs**

In [91]:
!pip -q install pdfplumber pandas nltk tqdm

# **Imports**

In [92]:
import re
import json
import random
from pathlib import Path

import pdfplumber
import pandas as pd
from tqdm import tqdm

import nltk
nltk.download("punkt")
nltk.download("punkt_tab")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [93]:
# ====== CONFIG ======
RAW_PDF_DIR = Path("/content/drive/MyDrive/DSA4265_Project/Transcripts")   # your uploaded PDFs are here
PROJECT_DIR = Path("/content/drive/MyDrive/DSA4265_Project/Results")

RAW_TEXT_DIR = PROJECT_DIR / "raw_text"
CLEAN_TXT_DIR = PROJECT_DIR / "cleaned_transcripts"
SENTENCE_DIR = PROJECT_DIR / "sentence_data"
CHUNK_DIR = PROJECT_DIR / "chunk_data"
RETRIEVER_DIR = PROJECT_DIR / "retriever_data"
LLM_DIR = PROJECT_DIR / "llm_data"

for d in [RAW_TEXT_DIR, CLEAN_TXT_DIR, SENTENCE_DIR, CHUNK_DIR, RETRIEVER_DIR, LLM_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Folders ready:", PROJECT_DIR)

Folders ready: /content/drive/MyDrive/DSA4265_Project/Results


In [94]:
pdf_files = sorted(RAW_PDF_DIR.glob("*.pdf"))
for f in pdf_files:
    print(f.name)

Amazon Q3 2025 Transcript.pdf
Amazon Q4 2025 Transcript.pdf
Apple Q3 2025 Transcript.pdf
Apple Q4 2025 Transcript.pdf
Meta Q3 2025 Transcript.pdf
Meta Q4 2025 Transcript.pdf
Tesla Q3 2025 Transcript.pdf
Tesla Q4 2025 Transcript.pdf


# **Helper Functions: Parse Filename**

In [95]:
def parse_company_quarter(filename: str):
    """
    Supports both:
    1. Original PDF names:
       - Amazon Q3 2025 Transcript.pdf
       - Apple Q4 2025 Transcript.pdf

    2. Processed raw text names:
       - amazon_Q3_2025_raw.txt
       - apple_Q4_2025_raw.txt
       - meta_Q3_2025.txt
    """
    stem = Path(filename).stem.strip()

    # Pattern 1: original PDF-style names
    m1 = re.match(r"([A-Za-z]+)\s+Q([1-4])\s+(\d{4})", stem, flags=re.IGNORECASE)
    if m1:
        company = m1.group(1).strip().lower()
        quarter = f"Q{m1.group(2)}"
        year = m1.group(3)
        return {
            "company": company,
            "quarter": quarter,
            "year": year,
            "company_quarter": f"{company}_{quarter}_{year}"
        }

    # Pattern 2: underscore-style processed names
    m2 = re.match(r"([A-Za-z]+)_Q([1-4])_(\d{4})(?:_raw)?", stem, flags=re.IGNORECASE)
    if m2:
        company = m2.group(1).strip().lower()
        quarter = f"Q{m2.group(2)}"
        year = m2.group(3)
        return {
            "company": company,
            "quarter": quarter,
            "year": year,
            "company_quarter": f"{company}_{quarter}_{year}"
        }

    return {
        "company": "unknown",
        "quarter": "unknown",
        "year": "unknown",
        "company_quarter": "unknown"
    }

# **Extract Raw Text**

In [96]:
def extract_pdf_text(pdf_path: Path) -> str:
    pages = []
    with pdfplumber.open(pdf_path) as pdf:
        for page_num, page in enumerate(pdf.pages, start=1):
            text = page.extract_text()
            if text:
                pages.append(text)
    return "\n".join(pages)

# **Save extracted raw text**

In [97]:
for pdf_file in tqdm(pdf_files, desc="Extracting raw text"):
    meta = parse_company_quarter(pdf_file.name)
    raw_text = extract_pdf_text(pdf_file)

    out_path = RAW_TEXT_DIR / f"{meta['company_quarter']}_raw.txt"
    out_path.write_text(raw_text, encoding="utf-8")

print("Raw text extraction complete.")

Extracting raw text: 100%|██████████| 8/8 [01:47<00:00, 13.43s/it]

Raw text extraction complete.


# **Cleaning Functions**

In [98]:
SPEAKER_PATTERN = re.compile(r"^([A-Z][A-Za-z\.\-\'& ,]{1,80}):\s*(.*)$")

BAD_SECTION_HEADERS = [
    "Takeaways",
    "Risks",
    "Summary",
    "Industry glossary",
    "Call participants",
    "Call Participants",
    "DATE",
    "Date"
]

def normalize_whitespace(text: str) -> str:
    text = text.replace("\r", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

def cut_to_transcript_only(text: str) -> str:
    """
    Keep only content after 'Full Conference Call Transcript'
    """
    split_key = "Full Conference Call Transcript"
    if split_key in text:
        text = text.split(split_key, 1)[1]
    return text

def remove_boilerplate(text: str) -> str:
    # remove known promo / irrelevant lines
    text = re.sub(r"Need a quote from .*?(?:\n|$)", "\n", text, flags=re.IGNORECASE)
    text = re.sub(r"Email .*?(?:\n|$)", "\n", text, flags=re.IGNORECASE)

    # remove common headings if they remain
    for h in BAD_SECTION_HEADERS:
        text = re.sub(rf"^\s*{re.escape(h)}\s*$", "", text, flags=re.MULTILINE)

    # remove bracketed stage instructions if you want cleaner text
    text = re.sub(r"\[.*?\]", "", text)

    # remove page artifacts like isolated numbers
    text = re.sub(r"^\s*\d+\s*$", "", text, flags=re.MULTILINE)

    return text

def clean_and_group_speaker_turns(text: str) -> list:
    """
    Returns list of dicts:
    [
      {"speaker": "Tim Cook", "text": "..."},
      ...
    ]
    """
    text = normalize_whitespace(text)
    text = cut_to_transcript_only(text)
    text = remove_boilerplate(text)
    text = normalize_whitespace(text)

    lines = [line.strip() for line in text.split("\n") if line.strip()]

    speaker_turns = []
    current_speaker = None
    current_parts = []

    for line in lines:
        match = SPEAKER_PATTERN.match(line)

        if match:
            # flush previous turn
            if current_speaker is not None:
                joined = " ".join(current_parts).strip()
                joined = re.sub(r"\s+", " ", joined)
                if joined:
                    speaker_turns.append({
                        "speaker": current_speaker,
                        "text": joined
                    })

            current_speaker = match.group(1).strip()
            first_text = match.group(2).strip()
            current_parts = [first_text] if first_text else []

        else:
            # continue previous speaker block
            if current_speaker is not None:
                current_parts.append(line)

    # flush last turn
    if current_speaker is not None:
        joined = " ".join(current_parts).strip()
        joined = re.sub(r"\s+", " ", joined)
        if joined:
            speaker_turns.append({
                "speaker": current_speaker,
                "text": joined
            })

    return speaker_turns

# **Cleaning + Save Cleaned Text**

In [99]:
all_turn_rows = []

for raw_txt_file in tqdm(sorted(RAW_TEXT_DIR.glob("*_raw.txt")), desc="Cleaning transcripts"):
    meta = parse_company_quarter(raw_txt_file.name)
    raw_text = raw_txt_file.read_text(encoding="utf-8")

    speaker_turns = clean_and_group_speaker_turns(raw_text)

    # save clean text file
    out_lines = []
    for turn in speaker_turns:
        out_lines.append(f"[Speaker: {turn['speaker']}] {turn['text']}")

    clean_text = "\n".join(out_lines)
    out_path = CLEAN_TXT_DIR / f"{meta['company_quarter']}.txt"
    out_path.write_text(clean_text, encoding="utf-8")

    # save structured rows too
    for idx, turn in enumerate(speaker_turns):
        all_turn_rows.append({
            "company": meta["company"],
            "quarter": meta["quarter"],
            "year": meta["year"],
            "company_quarter": meta["company_quarter"],
            "turn_id": idx,
            "speaker": turn["speaker"],
            "text": turn["text"]
        })

turns_df = pd.DataFrame(all_turn_rows)
turns_df.to_csv(PROJECT_DIR / "all_speaker_turns.csv", index=False)
print(turns_df.head())
print("Total speaker turns:", len(turns_df))

Cleaning transcripts: 100%|██████████| 8/8 [00:00<00:00, 18.88it/s]

  company quarter  year company_quarter  turn_id         speaker  \
0  amazon      Q3  2025  amazon_Q3_2025        0     Dave Fildes   
1  amazon      Q3  2025  amazon_Q3_2025        1    Andrew Jassy   
2  amazon      Q3  2025  amazon_Q3_2025        2  Brian Olsavsky   
3  amazon      Q3  2025  amazon_Q3_2025        3        Operator   
4  amazon      Q3  2025  amazon_Q3_2025        4     Justin Post   

                                                text  
0  Hello, and welcome to our Q3 2025 financial re...  
1  Thanks, Dave. We saw strong growth across our ...  
2  Thanks, Andy. Starting with our top line finan...  
3  And our first question comes from the line of ...  
4  I'll ask on AWS. Can you just kind of go throu...  
Total speaker turns: 389


# **Inspect 1 cleaned file**

In [100]:
sample_file = sorted(CLEAN_TXT_DIR.glob("*.txt"))[0]
print(sample_file.name)
print()
print(sample_file.read_text(encoding="utf-8")[:4000])

amazon_Q3_2025.txt

[Speaker: Dave Fildes] Hello, and welcome to our Q3 2025 financial results conference call. Joining us today to answer your questions is Andy Jassy, our CEO; and Brian Olsavsky, our CFO. As you listen to today's conference call, we encourage you to have our press release in front of you, which includes our financial results as well as metrics and commentary on the quarter. Please note, unless otherwise stated. All comparisons in this call will be against our results for the comparable period of 2024. Our comments and responses to your questions reflect management's views as of today, October 30, 2025 only, and will include forward-looking statements. Actual results may differ materially. Additional information about factors that could potentially impact our financial results is included in today's press release and our filings with the SEC, including our most recent annual report on Form 10-K and subsequent filings. During this call, we may discuss certain non-GAAP 

# **Sentence splitting for BERT**

In [101]:
from nltk.tokenize import sent_tokenize

sentence_rows = []

for _, row in tqdm(turns_df.iterrows(), total=len(turns_df), desc="Sentence splitting"):
    sentences = sent_tokenize(row["text"])

    for sent_id, sent in enumerate(sentences):
        sent = sent.strip()
        sent = re.sub(r"\s+", " ", sent)

        # keep only meaningful sentences
        if len(sent) >= 20:
            sentence_rows.append({
                "company": row["company"],
                "quarter": row["quarter"],
                "year": row["year"],
                "company_quarter": row["company_quarter"],
                "turn_id": row["turn_id"],
                "sentence_id": sent_id,
                "speaker": row["speaker"],
                "sentence": sent
            })

sent_df = pd.DataFrame(sentence_rows)
sent_df.to_csv(SENTENCE_DIR / "sentences.csv", index=False)

print(sent_df.head())
print("Total sentences:", len(sent_df))

Sentence splitting: 100%|██████████| 389/389 [00:00<00:00, 1025.46it/s]


  company quarter  year company_quarter  turn_id  sentence_id      speaker  \
0  amazon      Q3  2025  amazon_Q3_2025        0            0  Dave Fildes   
1  amazon      Q3  2025  amazon_Q3_2025        0            1  Dave Fildes   
2  amazon      Q3  2025  amazon_Q3_2025        0            2  Dave Fildes   
3  amazon      Q3  2025  amazon_Q3_2025        0            3  Dave Fildes   
4  amazon      Q3  2025  amazon_Q3_2025        0            4  Dave Fildes   

                                            sentence  
0  Hello, and welcome to our Q3 2025 financial re...  
1  Joining us today to answer your questions is A...  
2  As you listen to today's conference call, we e...  
3              Please note, unless otherwise stated.  
4  All comparisons in this call will be against o...  
Total sentences: 3539


# **Labelling File for BERT sentiment**

In [102]:
# This is the file your team will manually label for BERT sentiment
# label choices later can be: positive / neutral / negative

bert_label_df = sent_df.copy()
bert_label_df["sentiment_label"] = ""   # leave blank for manual labeling
bert_label_df["notes"] = ""

bert_label_path = SENTENCE_DIR / "bert_sentiment_labeling_template.csv"
bert_label_df.to_csv(bert_label_path, index=False)

print("Saved:", bert_label_path)
bert_label_df.head()

Saved: /content/drive/MyDrive/DSA4265_Project/Results/sentence_data/bert_sentiment_labeling_template.csv


,company,quarter,year,company_quarter,turn_id,sentence_id,speaker,sentence,sentiment_label,notes
0,amazon,Q3,2025,amazon_Q3_2025,0,0,Dave Fildes,"Hello, and welcome to our Q3 2025 financial re...",,
1,amazon,Q3,2025,amazon_Q3_2025,0,1,Dave Fildes,Joining us today to answer your questions is A...,,
2,amazon,Q3,2025,amazon_Q3_2025,0,2,Dave Fildes,"As you listen to today's conference call, we e...",,
3,amazon,Q3,2025,amazon_Q3_2025,0,3,Dave Fildes,"Please note, unless otherwise stated.",,
4,amazon,Q3,2025,amazon_Q3_2025,0,4,Dave Fildes,All comparisons in this call will be against o...,,


# **Chunking for RAG (speaker-turned-based)**

In [103]:
def create_chunks_from_turns(df, turns_per_chunk=3, overlap=1):
    """
    Chunk by speaker turns.
    Example:
    chunk 1 = turns 0,1,2
    chunk 2 = turns 2,3,4   (if overlap=1)
    """
    chunk_rows = []

    for company_quarter, group in df.groupby("company_quarter"):
        group = group.sort_values("turn_id").reset_index(drop=True)

        company = group.loc[0, "company"]
        quarter = group.loc[0, "quarter"]
        year = group.loc[0, "year"]

        step = max(1, turns_per_chunk - overlap)
        chunk_id = 0

        for start in range(0, len(group), step):
            end = start + turns_per_chunk
            window = group.iloc[start:end]

            if len(window) == 0:
                continue

            chunk_text_parts = []
            speakers = []

            for _, row in window.iterrows():
                chunk_text_parts.append(f"[Speaker: {row['speaker']}] {row['text']}")
                speakers.append(row["speaker"])

            chunk_text = "\n".join(chunk_text_parts).strip()

            chunk_rows.append({
                "company": company,
                "quarter": quarter,
                "year": year,
                "company_quarter": company_quarter,
                "chunk_id": f"{company_quarter}_chunk_{chunk_id}",
                "start_turn_id": int(window["turn_id"].min()),
                "end_turn_id": int(window["turn_id"].max()),
                "speakers_in_chunk": list(dict.fromkeys(speakers)),
                "chunk_text": chunk_text
            })

            chunk_id += 1

            if end >= len(group):
                break

    return pd.DataFrame(chunk_rows)

chunks_df = create_chunks_from_turns(turns_df, turns_per_chunk=3, overlap=1)
chunks_df.to_csv(CHUNK_DIR / "rag_chunks.csv", index=False)

print(chunks_df.head())
print("Total chunks:", len(chunks_df))

  company quarter  year company_quarter                chunk_id  \
0  amazon      Q3  2025  amazon_Q3_2025  amazon_Q3_2025_chunk_0   
1  amazon      Q3  2025  amazon_Q3_2025  amazon_Q3_2025_chunk_1   
2  amazon      Q3  2025  amazon_Q3_2025  amazon_Q3_2025_chunk_2   
3  amazon      Q3  2025  amazon_Q3_2025  amazon_Q3_2025_chunk_3   
4  amazon      Q3  2025  amazon_Q3_2025  amazon_Q3_2025_chunk_4   

   start_turn_id  end_turn_id                            speakers_in_chunk  \
0              0            2  [Dave Fildes, Andrew Jassy, Brian Olsavsky]   
1              2            4      [Brian Olsavsky, Operator, Justin Post]   
2              4            6        [Justin Post, Andrew Jassy, Operator]   
3              6            8        [Operator, Brian Nowak, Andrew Jassy]   
4              8           10     [Andrew Jassy, Operator, Douglas Anmuth]   

                                          chunk_text  
0  [Speaker: Dave Fildes] Hello, and welcome to o...  
1  [Speaker: Brian

# **Save chunks as JSONL**

In [104]:
chunk_jsonl_path = CHUNK_DIR / "rag_chunks.jsonl"

with open(chunk_jsonl_path, "w", encoding="utf-8") as f:
    for _, row in chunks_df.iterrows():
        record = row.to_dict()
        record["speakers_in_chunk"] = list(record["speakers_in_chunk"])
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print("Saved:", chunk_jsonl_path)

Saved: /content/drive/MyDrive/DSA4265_Project/Results/chunk_data/rag_chunks.jsonl


# **Make a simple retriever labeling template**

In [105]:
# This helps Member 2 later.
# You create a sheet where the team can write queries and map them to relevant chunk_ids.

retriever_template = chunks_df[[
    "company", "quarter", "year", "company_quarter", "chunk_id", "chunk_text"
]].copy()

retriever_template["query"] = ""
retriever_template["relevance_label"] = ""   # relevant / not_relevant
retriever_template["notes"] = ""

retriever_template_path = RETRIEVER_DIR / "retriever_labeling_template.csv"
retriever_template.to_csv(retriever_template_path, index=False)

print("Saved:", retriever_template_path)
retriever_template.head()

Saved: /content/drive/MyDrive/DSA4265_Project/Results/retriever_data/retriever_labeling_template.csv


,company,quarter,year,company_quarter,chunk_id,chunk_text,query,relevance_label,notes
0,amazon,Q3,2025,amazon_Q3_2025,amazon_Q3_2025_chunk_0,"[Speaker: Dave Fildes] Hello, and welcome to o...",,,
1,amazon,Q3,2025,amazon_Q3_2025,amazon_Q3_2025_chunk_1,"[Speaker: Brian Olsavsky] Thanks, Andy. Starti...",,,
2,amazon,Q3,2025,amazon_Q3_2025,amazon_Q3_2025_chunk_2,[Speaker: Justin Post] I'll ask on AWS. Can yo...,,,
3,amazon,Q3,2025,amazon_Q3_2025,amazon_Q3_2025_chunk_3,[Speaker: Operator] And the next question come...,,,
4,amazon,Q3,2025,amazon_Q3_2025,amazon_Q3_2025_chunk_4,"[Speaker: Andrew Jassy] Yes. Well, first of al...",,,


# **Create starter query set for retrieval**

In [106]:
starter_queries = [
    "What were the key highlights from this earnings call?",
    "What risks did management mention?",
    "What was management's outlook for the next quarter?",
    "What did the company say about revenue growth?",
    "What did the company say about margins or profitability?",
    "What did management say about demand trends?",
    "What did the company say about AI investments or strategy?",
    "What concerns were raised during the Q&A?"
]

query_rows = []

for cq in sorted(chunks_df["company_quarter"].unique()):
    meta_rows = chunks_df[chunks_df["company_quarter"] == cq]
    company = meta_rows.iloc[0]["company"]
    quarter = meta_rows.iloc[0]["quarter"]
    year = meta_rows.iloc[0]["year"]

    for q in starter_queries:
        query_rows.append({
            "company": company,
            "quarter": quarter,
            "year": year,
            "company_quarter": cq,
            "query": q,
            "positive_chunk_id": "",   # fill manually
            "hard_negative_chunk_id": "",  # fill manually
            "notes": ""
        })

query_df = pd.DataFrame(query_rows)
query_df.to_csv(RETRIEVER_DIR / "starter_query_pairs_template.csv", index=False)

print(query_df.head())
print("Saved starter retrieval template.")

  company quarter  year company_quarter  \
0  amazon      Q3  2025  amazon_Q3_2025   
1  amazon      Q3  2025  amazon_Q3_2025   
2  amazon      Q3  2025  amazon_Q3_2025   
3  amazon      Q3  2025  amazon_Q3_2025   
4  amazon      Q3  2025  amazon_Q3_2025   

                                               query positive_chunk_id  \
0  What were the key highlights from this earning...                     
1                 What risks did management mention?                     
2  What was management's outlook for the next qua...                     
3     What did the company say about revenue growth?                     
4  What did the company say about margins or prof...                     

  hard_negative_chunk_id notes  
0                               
1                               
2                               
3                               
4                               
Saved starter retrieval template.


# **Create starter LLM SFT template**

In [107]:
# This is for Member 4 later, but you prepare the template.
# The team can fill in high-quality target summaries.

llm_rows = []

for company_quarter, group in chunks_df.groupby("company_quarter"):
    company = group.iloc[0]["company"]
    quarter = group.iloc[0]["quarter"]
    year = group.iloc[0]["year"]

    # take first few chunks as starter context sample
    context_chunks = group.head(3)["chunk_text"].tolist()
    context = "\n\n".join(context_chunks)

    llm_rows.append({
        "company": company,
        "quarter": quarter,
        "year": year,
        "company_quarter": company_quarter,
        "instruction": "Summarize this earnings call into Key Highlights, Risks Mentioned, Management Outlook, and Overall Tone.",
        "input_context": context,
        "target_output": ""   # to be manually written / refined by the team
    })

llm_df = pd.DataFrame(llm_rows)
llm_df.to_csv(LLM_DIR / "llm_sft_template.csv", index=False)

print(llm_df.head())
print("Saved:", LLM_DIR / "llm_sft_template.csv")

  company quarter  year company_quarter  \
0  amazon      Q3  2025  amazon_Q3_2025   
1  amazon      Q4  2025  amazon_Q4_2025   
2   apple      Q3  2025   apple_Q3_2025   
3   apple      Q4  2025   apple_Q4_2025   
4    meta      Q3  2025    meta_Q3_2025   

                                         instruction  \
0  Summarize this earnings call into Key Highligh...   
1  Summarize this earnings call into Key Highligh...   
2  Summarize this earnings call into Key Highligh...   
3  Summarize this earnings call into Key Highligh...   
4  Summarize this earnings call into Key Highligh...   

                                       input_context target_output  
0  [Speaker: Dave Fildes] Hello, and welcome to o...                
1  [Speaker: Dave Fildes] Hello, and welcome to o...                
2  [Speaker: Timothy D. Cook] Thank you, Suhasini...                
3  [Speaker: Timothy Cook] Thank you, Suhasini. G...                
4  [Speaker: Mark Zuckerberg] We had another stro...       

# **Qucik dataset summary**

In [108]:
summary = {
    "num_transcripts_cleaned": turns_df["company_quarter"].nunique(),
    "num_speaker_turns": len(turns_df),
    "num_sentences": len(sent_df),
    "num_chunks": len(chunks_df),
    "companies": sorted(turns_df["company"].unique().tolist()),
    "quarters": sorted(turns_df["quarter"].unique().tolist())
}

print(json.dumps(summary, indent=2))

{
  "num_transcripts_cleaned": 8,
  "num_speaker_turns": 389,
  "num_sentences": 3539,
  "num_chunks": 192,
  "companies": [
    "amazon",
    "apple",
    "meta",
    "tesla"
  ],
  "quarters": [
    "Q3",
    "Q4"
  ]
}


# **Zip all prepared data**

In [109]:
import shutil

zip_base = "/mnt/data/member1_data_prep_outputs"
shutil.make_archive(zip_base, "zip", PROJECT_DIR)

print("Saved zip:", zip_base + ".zip")

Saved zip: /mnt/data/member1_data_prep_outputs.zip
